In [2]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path

In [3]:
load_dotenv(Path.cwd().parent / ".env")

False

In [4]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [5]:
#Company House

def get(endpoint, params=None, headers=None):
    url = f"{URL_BASE}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(response.status_code)
    return response.json()

def get_meta_data(endpoint, params=None, headers=None):
    url = f"{URL_META_DATA}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    return response

In [6]:
import time, threading, itertools, re
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x            # no-op fallback if tqdm isn't installed

# --- Multi-key throttled Companies House client (uses all API keys -> ~Nx faster)
# CH's 600-requests/5-min limit is PER API KEY. We load every CH_API* key, give
# each its own Session + pacer (~1.7 req/s), and spread requests across them via a
# thread pool. Aggregate throughput ~= N_keys * 1.7 req/s (4 keys ~= 6.7 req/s).

def _clean_key(v):
    """Extract a key value written as  KEY = 'abc',  /  "abc",  /  abc  (the .env
       here stores them Python-list style with quotes and trailing commas)."""
    v = v.strip()
    m = re.search(r"""['"]([^'"]+)['"]""", v)   # value inside the first quote pair
    return m.group(1) if m else v.rstrip(",").strip()


def _load_ch_keys(names=("CH_API", "CH_API_2", "CH_API_3", "CH_API_4")):
    """Load keys tolerantly from the repo-root .env (handles spaces around '=',
       quotes and trailing commas that python-dotenv rejects), then env vars."""
    parsed = {}
    for cand in (Path.cwd().parent.parent / ".env",
                 Path.cwd().parent / ".env",
                 Path.cwd() / ".env"):
        if cand.exists():
            for line in cand.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                parsed[k.strip()] = _clean_key(v)
            break
    keys = [parsed.get(n) or os.getenv(n) for n in names]
    return [k for k in keys if k]


_KEYS = _load_ch_keys()
if not _KEYS:
    raise RuntimeError("No Companies House API keys found (checked .env and env vars)")

_MIN_INTERVAL = 0.6           # seconds between calls PER KEY (~1.7/s; CH cap 2.0/s each)
MAX_WORKERS   = 2 * len(_KEYS)
print(f"{len(_KEYS)} API key(s) loaded -> up to ~{len(_KEYS) / _MIN_INTERVAL:.1f} req/s "
      f"with {MAX_WORKERS} workers")


class _Lane:
    """One API key: dedicated Session + a lock enforcing min-interval pacing."""
    __slots__ = ("session", "lock", "last")
    def __init__(self, key):
        self.session = requests.Session(); self.session.auth = (key, "")
        self.lock = threading.Lock(); self.last = 0.0
    def pace(self):
        with self.lock:
            wait = _MIN_INTERVAL - (time.monotonic() - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.monotonic()


_LANES = [_Lane(k) for k in _KEYS]
_rr = itertools.count(); _rr_lock = threading.Lock()

def _next_lane():
    with _rr_lock:
        i = next(_rr)
    return _LANES[i % len(_LANES)]


def ch_get(path, params=None, max_retries=5):
    """Thread-safe, key-rotating, throttled GET. Returns a Response (200/404/etc.
       for the caller to interpret) or None if it gave up. Handles 429 (waits out
       the window) and transient network/5xx errors with exponential backoff."""
    url = f"{URL_BASE}{path}"
    for attempt in range(max_retries):
        lane = _next_lane()
        lane.pace()
        try:
            r = lane.session.get(url, params=params, timeout=TIMEOUT)
        except (ConnectionError, Timeout):
            time.sleep(2 ** attempt); continue
        if r.status_code == 429:                       # rare with per-key pacing
            time.sleep(int(r.headers.get("Retry-After", 60)) + 1); continue
        if r.status_code >= 500:                        # transient server error
            time.sleep(2 ** attempt); continue
        return r
    return None


def parallel_fill(df, todo_index, fn, cols, out_path,
                  desc="fetch", max_workers=None, checkpoint_every=200):
    """Concurrently apply fn(com_num) over todo_index rows and write results into
       `cols` (str, or list matching fn's tuple return). Only network I/O runs in
       threads; all DataFrame writes + checkpoints happen on the main thread, so no
       df locking is needed. Resumable + checkpointed + live rate/ETA."""
    cols = [cols] if isinstance(cols, str) else list(cols)
    todo_index = list(todo_index)
    n_todo = len(todo_index)
    workers = max_workers or MAX_WORKERS
    print(f"{n_todo} to fetch (of {len(df)}) — {len(_LANES)} keys x {workers} workers")
    t0 = time.time(); done = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        fut2idx = {ex.submit(fn, df.at[idx, "com_num"]): idx for idx in todo_index}
        for fut in tqdm(as_completed(fut2idx), total=n_todo, desc=desc):
            idx = fut2idx[fut]
            try:
                res = fut.result()
            except Exception:
                res = "error" if len(cols) == 1 else tuple("error" for _ in cols)
            if len(cols) == 1:
                df.at[idx, cols[0]] = res
            else:
                for c, v in zip(cols, res):
                    df.at[idx, c] = v
            done += 1
            if done % checkpoint_every == 0:
                df.to_csv(out_path, index=False)
                rate = done / (time.time() - t0)
                eta = (n_todo - done) / rate / 60 if rate else 0
                print(f"  {done}/{n_todo} | {rate:.2f}/s | ETA ~{eta:.0f} min | checkpoint saved")
    df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")

4 API key(s) loaded -> up to ~6.7 req/s with 8 workers


## Stage 1: profile pull — `account_type` + `accounts_overdue` + panel fields (one call)

Scans **every** CSV in `company_data/company_raw/`, dedupes on `com_num`, and pulls the company profile **only for new company numbers**. One `/company/{num}` call now yields **six** columns — the original two plus the point-in-time **panel fields**, at zero extra API cost:

- `account_type`, `accounts_overdue` — as before
- `date_of_creation` — incorporation date → `age@T` for the panel
- `company_status` — active / dissolved / liquidation (drop non-active from lead lists)
- `last_accounts_made_up_to`, `next_accounts_due_on` — point-in-time accounts recency / punctuality

Companies pulled **before** this change only have the first two — **Stage 5** backfills the rest one-time; new companies never need that second call.

Output → `company_data/company_with_acctype/com_names_with_acctype_test.csv`

In [7]:
# --- Stage 1: ONE profile call per NEW company -> 6 columns ---------------------
# Scans every CSV in company_raw/, dedupes on com_num, and fetches the profile
# ONLY for company numbers not already in the output. ALL six columns come from
# the same /company/{num} response, so the panel fields (date_of_creation,
# company_status, accounts dates) are captured at ZERO extra API cost. Companies
# pulled before this change lack them -> Stage 5 backfills those one-time.
BASE        = Path("company_data")
RAW_DIR     = BASE / "company_raw"
ACCTYPE_OUT = BASE / "company_with_acctype" / "com_names_with_acctype_test.csv"

PROFILE_COLS = ["account_type", "accounts_overdue",                 # original two
                "date_of_creation", "company_status",               # panel fields
                "last_accounts_made_up_to", "next_accounts_due_on"]

# 1. Combine every raw file; one row per company (raw files repeat com_num per SIC).
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")

# 2. Load prior results; NEW companies = those not already processed.
if ACCTYPE_OUT.exists():
    done = pd.read_csv(ACCTYPE_OUT, dtype=str)
else:
    done = pd.DataFrame(columns=list(raw.columns))
for c in PROFILE_COLS:
    if c not in done.columns:
        done[c] = pd.NA          # legacy rows: panel fields backfilled in Stage 5

new = raw[~raw["com_num"].isin(done["com_num"])].copy()
for c in PROFILE_COLS:
    new[c] = pd.NA
df = pd.concat([done, new], ignore_index=True)   # keep old results, append new rows
df = df.drop(columns=[c for c in ["com_type", "town", "county"]  # unused raw passthrough
                      if c in df.columns])


def fetch_profile_fields(company_num):
    """All six columns from ONE profile call.
       Sentinels: 'no_accounts', 'not_found', 'error', 'unknown', 'none'."""
    r = ch_get(f"/company/{str(company_num).strip()}")
    if r is None:              return ("error",) * 6
    if r.status_code == 404:   return ("not_found",) * 6
    if r.status_code != 200:   return ("error",) * 6
    j = r.json()
    acc = j.get("accounts", {})
    la  = acc.get("last_accounts", {})
    ov  = acc.get("overdue")
    return (la.get("type") or "no_accounts",
            "unknown" if ov is None else str(ov),
            j.get("date_of_creation") or "unknown",
            j.get("company_status") or "unknown",
            la.get("made_up_to") or "none",
            acc.get("next_accounts", {}).get("due_on") or "none")


# Fetch only rows missing account_type (new companies + any interrupted ones).
todo = df.index[df["account_type"].isna()]
print(f"{len(new)} new companies from raw | {len(done)} already done")
parallel_fill(df, todo, fetch_profile_fields, PROFILE_COLS, ACCTYPE_OUT, desc="profile")
df["account_type"].value_counts(dropna=False)

0 new companies from raw | 102039 already done
18 to fetch (of 102039) — 4 keys x 8 workers
Saved -> company_data/company_with_acctype/com_names_with_acctype_test.csv


account_type
micro-entity                   42304
total-exemption-full           25973
no_accounts                    20742
dormant                         7241
unaudited-abridged              3135
full                             884
small                            769
audit-exemption-subsidiary       485
group                            284
medium                            97
total-exemption-small             71
no-accounts-type-available        24
null                              18
audited-abridged                   9
filing-exemption-subsidiary        3
Name: count, dtype: int64

In [8]:
# --- Remove perfect duplicate rows from the acctype file -----------------------
# A "perfect duplicate" = every column identical. The pull already dedupes on
# com_num, so this is now a harmless safety net (expect 0 removed).
dedup = pd.read_csv(ACCTYPE_OUT, dtype=str)
before = len(dedup)
dedup = dedup.drop_duplicates()          # all columns must match
removed = before - len(dedup)

dedup.to_csv(ACCTYPE_OUT, index=False)
print(f"Removed {removed} perfect duplicate row(s): {before} -> {len(dedup)} rows in {ACCTYPE_OUT}")


Removed 0 perfect duplicate row(s): 102039 -> 102039 rows in company_data/company_with_acctype/com_names_with_acctype_test.csv


## Stage 2: filter to SME account types (no API calls)

Keep only companies whose `account_type` is one of `small`, `micro-entity`, `total-exemption-full`, `total-exemption-small`, `medium`. Carries forward **every** already-computed enrichment column from the existing SME file, so stage 3 only fetches genuinely new companies. Output → `com_names_sme.csv`.

In [9]:
# --- Stage 2: keep only SME-relevant account types (no API calls) ---------------
# Rebuilds the SME set from the acctype file and CARRIES FORWARD every enrichment
# column already computed in the existing SME file -- so stage 3 fetches only the
# genuinely new SME companies. The panel fields (Stage 1 for new companies /
# Stage 5 backfill for old ones) and n_charges (Stage 4) are in the carry list so
# a rebuild never wipes them.
SME_TYPES = ["small", "micro-entity", "total-exemption-full",
             "total-exemption-small", "medium"]
ENRICH_COLS = ["lloyds_customer", "recent_psc_kind",
               "recent_charge_status", "recent_charge_created_on", "accounts_overdue",
               # panel fields + charge-history bookkeeping (carried across rebuilds)
               "date_of_creation", "company_status",
               "last_accounts_made_up_to", "next_accounts_due_on", "n_charges"]

BASE        = Path("company_data")
ACCTYPE_OUT = BASE / "company_with_acctype" / "com_names_with_acctype_test.csv"
SME_OUT     = BASE / "company_sme_with_acctype" / "com_names_sme_test.csv"

full = pd.read_csv(ACCTYPE_OUT, dtype=str)
sme  = full[full["account_type"].isin(SME_TYPES)].copy()

if SME_OUT.exists():
    prev = pd.read_csv(SME_OUT, dtype=str)
    keep = ["com_num"] + [c for c in ENRICH_COLS if c in prev.columns]
    sme = sme.merge(prev[keep], on="com_num", how="left", suffixes=("", "_prev"))
    # a column can exist in BOTH files (acctype captures it at stage 1 for new
    # companies; the SME file has it for old ones) -> prefer fresh, else prev.
    for c in ENRICH_COLS:
        if f"{c}_prev" in sme.columns:
            sme[c] = sme[c].fillna(sme[f"{c}_prev"])
            sme.drop(columns=[f"{c}_prev"], inplace=True)
for c in ENRICH_COLS:
    if c not in sme.columns:
        sme[c] = pd.NA
sme = sme.drop(columns=[c for c in ["com_type", "town", "county"] if c in sme.columns])

sme.to_csv(SME_OUT, index=False)
print(f"SME companies: {len(sme)} of {len(full)} -> {SME_OUT}")
pending = sme[["lloyds_customer", "recent_psc_kind", "recent_charge_status",
               "recent_charge_created_on", "accounts_overdue"]].isna().any(axis=1).sum()
print(f"  enrichment pending (stage 3) for {pending} companies")
sme["account_type"].value_counts()

SME companies: 69214 of 102039 -> company_data/company_sme_with_acctype/com_names_sme_test.csv
  enrichment pending (stage 3) for 0 companies


account_type
micro-entity             42304
total-exemption-full     25973
small                      769
medium                      97
total-exemption-small       71
Name: count, dtype: int64

## Stage 3: SME enrichment — one charges call + one PSC call per company

For each new SME company, **2 API calls** fill **4 columns**:
- `/charges` (paged once) → `lloyds_customer` **and** `recent_charge_status` + `recent_charge_created_on` (previously two separate charges passes)
- `/persons-with-significant-control` → `recent_psc_kind`

`lloyds_customer` matches the whole Lloyds Banking Group ("Lloyds", Bank of Scotland, HBOS, Halifax, AMC, …) but **not** "Lloyd's" of London. "Most recent" = latest `created_on` / `notified_on`. Resumable — only rows missing any of the four are fetched.

**Since the panel work:** the charges call now also **persists the FULL charge list** to `company_info_json/{com_num}_charges.json` while it's already in memory — **zero extra API calls**. That JSON is the raw source for the per-charge history table (**Stage 4**): every charge's `created_on` + lender + status, which the panel needs for the label (Lloyds charge after cutoff T), the existing-customer exclusion (Lloyds charge ≤ T), and the prior-non-Lloyds-borrowing feature (≤ T). Companies enriched *before* this change have no JSON yet — Stage 4 backfills only those.

In [10]:
import re

# --- Stage 3: SME enrichment (multi-key concurrent) ------------------------------
# ONE charges call (paged) + ONE PSC call per company fill FOUR columns:
#   lloyds_customer            <- charges: any charge held by a Lloyds-group entity
#   recent_charge_status       <- charges: most recent charge (latest created_on)
#   recent_charge_created_on   <- charges
#   recent_psc_kind            <- PSC: kind of the PSC with latest notified_on
# (accounts_overdue comes free with the stage-1 profile pull, not here.)
# Resumable: only processes rows still missing ANY of the four.
#
# PANEL: the full charge list is ALREADY in memory here, so we persist it to
# company_info_json/{com_num}_charges.json as we go -- zero extra API calls.
# Stage 4 parses those JSONs into the per-charge history table the panel needs.
#
# Lloyds matching: the token "Lloyds" (no apostrophe) is the bank; it deliberately
# does NOT match "Lloyd's" (apostrophe) = Lloyd's of London, the insurance market.
# List covers the whole Lloyds Banking Group; trim to r"\blloyds\b" for bank-only.
LLOYDS_PATTERNS = [
    r"\blloyds\b",                       # Lloyds Bank, Lloyds TSB, Lloyds Commercial Finance, Lloyds Dev. Capital
    r"bank of scotland",                 # Bank of Scotland plc (Halifax's legal entity too)
    r"\bhbos\b",                         # HBOS plc
    r"agricultural mortgage corp",       # AMC — Lloyds' farm/agri lender
    r"\bhalifax\b",                      # Halifax (division of Bank of Scotland)
    r"birmingham midshires",             # BM — mortgages
    r"cheltenham (?:& |and )gloucester", # C&G — mortgages
    r"bank of wales",                    # historic BoS brand
    r"black horse",                      # Lloyds motor/asset finance
    r"scottish widows",                  # pensions/insurance
    r"\bmbna\b",                         # credit cards (Lloyds acquired 2017)
]
LLOYDS_RE = re.compile("|".join(LLOYDS_PATTERNS), re.I)

SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme_test.csv"
ENRICH_COLS = ["lloyds_customer", "recent_charge_status",
               "recent_charge_created_on", "recent_psc_kind"]
CHARGE_JSON_DIR = Path("company_info_json")      # full charge history persisted here
CHARGE_JSON_DIR.mkdir(exist_ok=True)

sme = pd.read_csv(SME_OUT, dtype=str)
for col in ENRICH_COLS:
    if col not in sme.columns:
        sme[col] = pd.NA


def enrich_sme(company_num):
    """(lloyds_customer, recent_charge_status, recent_charge_created_on,
        recent_psc_kind) — one paged charges call + one PSC call. Also persists
        the FULL charge list to JSON (free: it's already in memory)."""
    cnum = str(company_num).strip()

    # -- charges: single pass yields BOTH the Lloyds flag and the latest charge --
    lloyds, status, created = "False", None, None
    start, charges = 0, []
    while True:
        r = ch_get(f"/company/{cnum}/charges",
                   params={"items_per_page": 100, "start_index": start})
        if r is None:            lloyds = status = created = "error";  break
        if r.status_code == 404: status = created = "no_charges";      break
        if r.status_code != 200: lloyds = status = created = "error";  break
        data = r.json()
        items = data.get("items", [])
        charges.extend(items)
        for ch in items:
            for p in ch.get("persons_entitled", []):
                if LLOYDS_RE.search(p.get("name", "")):
                    lloyds = "True"        # keep paging: still need the latest charge
        start += len(items)
        if not items or start >= data.get("total_count", 0):
            break
    if status is None:                      # paged through without error
        if charges:
            latest = max(charges, key=lambda c: c.get("created_on") or "")
            status = latest.get("status") or "unknown"
            created = latest.get("created_on") or "unknown"
        else:
            status = created = "no_charges"

    # -- persist the FULL charge list while we hold it (zero extra API calls) --
    # (404/no-charges companies get an empty items list = "pulled, none found",
    #  which lets Stage 4 skip them; error paths write nothing so they retry.)
    if status != "error":
        try:
            (CHARGE_JSON_DIR / f"{cnum}_charges.json").write_text(
                json.dumps({"total_count": len(charges), "items": charges}))
        except OSError:
            pass                            # persistence is best-effort

    # -- PSC: kind of the most recent PSC (latest notified_on) --
    rp = ch_get(f"/company/{cnum}/persons-with-significant-control",
                params={"items_per_page": 100})
    if rp is None:              psc = "error"
    elif rp.status_code == 404: psc = "no_psc"
    elif rp.status_code != 200: psc = "error"
    else:
        items = rp.json().get("items", [])
        if not items:
            psc = "no_psc"
        else:
            latest = max(items, key=lambda it: it.get("notified_on") or "")
            psc = latest.get("kind") or "unknown"

    return lloyds, status, created, psc


todo = sme.index[sme[ENRICH_COLS].isna().any(axis=1)]
parallel_fill(sme, todo, enrich_sme, ENRICH_COLS, SME_OUT, desc="sme_enrich")
sme[ENRICH_COLS].apply(lambda c: c.value_counts(dropna=False).head(6))

0 to fetch (of 69214) — 4 keys x 8 workers
Saved -> company_data/company_sme_with_acctype/com_names_sme_test.csv


,lloyds_customer,recent_charge_status,recent_charge_created_on,recent_psc_kind
2025-03-28,NaN,NaN,19.0,NaN
2026-02-06,NaN,NaN,22.0,NaN
2026-05-15,NaN,NaN,20.0,NaN
2026-06-19,NaN,NaN,17.0,NaN
2026-06-30,NaN,NaN,19.0,NaN
False,67048.0,NaN,NaN,NaN
True,2166.0,NaN,NaN,NaN
corporate-entity-person-with-significant-control,NaN,NaN,NaN,4338.0
fully-satisfied,NaN,2095.0,NaN,NaN
individual-person-with-significant-control,NaN,NaN,NaN,57663.0


## Stage 4: full charge history → `charges_history.csv` (the panel's label source)

The panel needs **every charge with its date and lender**, not just the recent-charge summary:
- **label** — Lloyds-group charge `created_on` **after** cutoff T (within the horizon) → positive
- **existing-customer exclusion** — Lloyds-group charge `created_on` **≤ T** → dropped from that cutoff
- **feature** — count of **non-Lloyds** charges `created_on` ≤ T (prior secured borrowing; allowed, not leakage)

**Efficiency — this costs ~1.8k calls, not 13.5k (~87% skipped):**
1. Stage 3 already told us which companies have **no charges** (`recent_charge_status == 'no_charges'`, ~11.8k of 13.5k) → **zero API calls** for them.
2. Companies whose JSON was already persisted (by the updated Stage 3, or a previous Stage 4 run) → **zero calls**.
3. Only companies **with** charges and **no JSON on disk** are fetched — ~**1,790** one-time (≈ 9 min at ~6.7 req/s across 4 keys). Resumable/checkpointed like every other stage.

Then a **zero-call parse** turns all persisted JSONs into one tidy table — one row per charge:
`com_num · charge_code · created_on · delivered_on · satisfied_on · status · classification · persons_entitled · is_lloyds`
(`is_lloyds` uses Stage 3's `LLOYDS_RE` — the curated Lloyds-group entity list.)

Output → `company_data/charges_history.csv`. ⚠️ Run the **Stage 3 cell first** (defines `LLOYDS_RE`; fills `recent_charge_status` for new companies).

In [11]:
# --- Stage 4: full charge history -> charges_history.csv -------------------------
# One-time backfill for companies enriched BEFORE Stage 3 started persisting the
# charge JSONs, then a zero-call parse of every JSON into one tidy per-charge table.
# Requires the Stage 3 cell to have run (LLOYDS_RE, CHARGE_JSON_DIR, SME_OUT).
CHARGES_OUT = Path("company_data") / "charges_history.csv"

sme = pd.read_csv(SME_OUT, dtype=str)
if "n_charges" not in sme.columns:
    sme["n_charges"] = pd.NA


def _charge_file(cnum):
    return CHARGE_JSON_DIR / f"{str(cnum).strip()}_charges.json"


# --- who needs an API call? -------------------------------------------------------
# no_charges/not_found (from stage 3)  -> 0 calls, mark n_charges=0 directly
# JSON already on disk                 -> 0 calls (stage 3 persisted it, or a
#                                         previous stage-4 run did)
# everyone else WITH charges           -> the one-time backfill (~1.8k companies)
no_need   = sme["recent_charge_status"].isin(["no_charges", "not_found"])
unknown   = sme["recent_charge_status"].isna()          # stage 3 not run for these
have_json = sme["com_num"].map(lambda c: _charge_file(c).exists())

sme.loc[no_need & sme["n_charges"].isna(), "n_charges"] = "0"   # free: no call needed
todo = sme.index[~no_need & ~unknown & ~have_json]
if unknown.any():
    print(f"NOTE: {unknown.sum()} companies have no recent_charge_status yet — "
          "run Stage 3 first so they can be classified (skipped here).")
print(f"charge history: {int(no_need.sum())} skipped as no-charges (0 calls) | "
      f"{int((~no_need & have_json).sum())} already on disk (0 calls) | "
      f"{len(todo)} to fetch")


def fetch_charge_history(company_num):
    """Page /charges and persist the FULL item list to JSON. Returns n_charges."""
    cnum = str(company_num).strip()
    start, items = 0, []
    while True:
        r = ch_get(f"/company/{cnum}/charges",
                   params={"items_per_page": 100, "start_index": start})
        if r is None:
            return "error"
        if r.status_code == 404:
            break                                    # no charges registered
        if r.status_code != 200:
            return "error"
        data = r.json()
        page = data.get("items", [])
        items.extend(page)
        start += len(page)
        if not page or start >= data.get("total_count", 0):
            break
    _charge_file(cnum).write_text(json.dumps({"total_count": len(items),
                                              "items": items}))
    return str(len(items))


parallel_fill(sme, todo, fetch_charge_history, "n_charges", SME_OUT, desc="charges_hist")

# --- parse ALL persisted JSONs -> one row per charge (zero API calls) -------------
rows = []
for cnum in sme["com_num"]:
    f = _charge_file(cnum)
    if not f.exists():
        continue
    for ch in json.loads(f.read_text()).get("items", []):
        lenders = " | ".join(p.get("name", "") for p in ch.get("persons_entitled", []))
        rows.append({
            "com_num":          cnum,
            "charge_code":      ch.get("charge_code") or ch.get("charge_number"),
            "created_on":       ch.get("created_on"),
            "delivered_on":     ch.get("delivered_on"),
            "satisfied_on":     ch.get("satisfied_on"),
            "status":           ch.get("status"),
            "classification":   (ch.get("classification") or {}).get("description"),
            "persons_entitled": lenders,
            "is_lloyds":        bool(LLOYDS_RE.search(lenders)),
        })
charges_hist = pd.DataFrame(rows)
charges_hist.to_csv(CHARGES_OUT, index=False)

# --- sanity report -----------------------------------------------------------------
n_cos = charges_hist["com_num"].nunique() if len(charges_hist) else 0
n_llo = int(charges_hist["is_lloyds"].sum()) if len(charges_hist) else 0
llo_cos = charges_hist.loc[charges_hist["is_lloyds"], "com_num"].nunique() if len(charges_hist) else 0
print(f"\ncharges_history: {len(charges_hist)} charges across {n_cos} companies "
      f"-> {CHARGES_OUT}")
print(f"  Lloyds-group charges: {n_llo} across {llo_cos} companies "
      f"(cf. lloyds_customer=True in the SME file: "
      f"{(sme['lloyds_customer'] == 'True').sum()})")
if len(charges_hist):
    print(f"  created_on range: {charges_hist['created_on'].min()} -> "
          f"{charges_hist['created_on'].max()}")
    print("  most common lenders:")
    print(charges_hist["persons_entitled"].value_counts().head(8).to_string())

charge history: 57478 skipped as no-charges (0 calls) | 11736 already on disk (0 calls) | 0 to fetch
0 to fetch (of 69214) — 4 keys x 8 workers
Saved -> company_data/company_sme_with_acctype/com_names_sme_test.csv

charges_history: 42003 charges across 11736 companies -> company_data/charges_history.csv
  Lloyds-group charges: 5849 across 2166 companies (cf. lloyds_customer=True in the SME file: 2166)
  created_on range: 1906-10-03 -> 2026-07-16
  most common lenders:
persons_entitled
National Westminster Bank PLC          3531
Barclays Bank PLC                      2865
Lloyds Bank PLC                        2206
The Royal Bank of Scotland PLC         1261
The Mortgage Works (UK) PLC            1252
Hsbc Bank PLC                          1198
Together Commercial Finance Limited    1178
Paragon Bank PLC                        903


## Stage 5: backfill panel fields — `date_of_creation` & co. (one-time)

Stage 1 now captures four **panel fields** from the profile call it already makes — but every company pulled *before* that change is missing them. This one-time backfill re-pulls the profile **only for SME rows missing `date_of_creation`**:

- `date_of_creation` — incorporation date → **`age@T`** (age at any cutoff = T − this date)
- `company_status` — active / dissolved / liquidation → drop non-active firms from lead lists
- `last_accounts_made_up_to` — most recent accounts date → accounts recency at T
- `next_accounts_due_on` — due date → punctuality / overdue-at-T

**Cost:** one `/company/{num}` call per legacy SME (~13.5k ≈ **34 min** at ~6.7 req/s across 4 keys) — one-time only. Resumable/checkpointed: interrupt any time and re-run; already-filled rows are never re-fetched, and **future runs cost ~0 calls** because Stage 1 captures these fields for every new company (Stage 2 carries them forward).

Writes into the SME file (`com_names_sme_test.csv`) in place — after this, the SME table + `charges_history.csv` (+ the GDELT `media_index`) are everything `build_panel()` needs.

In [12]:
# --- Stage 5: backfill panel fields from the profile (one-time, resumable) -------
# Only SME rows missing date_of_creation are fetched; new companies get these at
# Stage 1 for free, so this backfill shrinks to zero over time.
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme_test.csv"
PANEL_COLS = ["date_of_creation", "company_status",
              "last_accounts_made_up_to", "next_accounts_due_on"]

sme = pd.read_csv(SME_OUT, dtype=str)
for c in PANEL_COLS:
    if c not in sme.columns:
        sme[c] = pd.NA


def fetch_panel_fields(company_num):
    """(date_of_creation, company_status, last_accounts_made_up_to,
        next_accounts_due_on) from ONE profile call."""
    r = ch_get(f"/company/{str(company_num).strip()}")
    if r is None:              return ("error",) * 4
    if r.status_code == 404:   return ("not_found",) * 4
    if r.status_code != 200:   return ("error",) * 4
    j = r.json()
    acc = j.get("accounts", {})
    return (j.get("date_of_creation") or "unknown",
            j.get("company_status") or "unknown",
            acc.get("last_accounts", {}).get("made_up_to") or "none",
            acc.get("next_accounts", {}).get("due_on") or "none")


todo = sme.index[sme["date_of_creation"].isna()]
parallel_fill(sme, todo, fetch_panel_fields, PANEL_COLS, SME_OUT, desc="panel_fields")

# --- sanity report ----------------------------------------------------------------
print("\npanel-field coverage:")
for c in PANEL_COLS:
    filled = sme[c].notna() & ~sme[c].isin(["error", "not_found"])
    print(f"  {c:26s} {int(filled.sum())}/{len(sme)}")
if sme["date_of_creation"].notna().any():
    ok = sme.loc[~sme["date_of_creation"].isin(["error", "not_found", "unknown"]),
                 "date_of_creation"]
    print(f"\n  incorporation dates: {ok.min()} -> {ok.max()}")
    print("  company_status:")
    print(sme["company_status"].value_counts(dropna=False).head(6).to_string())

0 to fetch (of 69214) — 4 keys x 8 workers
Saved -> company_data/company_sme_with_acctype/com_names_sme_test.csv

panel-field coverage:
  date_of_creation           69214/69214
  company_status             69214/69214
  last_accounts_made_up_to   69214/69214
  next_accounts_due_on       69214/69214

  incorporation dates: 1887-03-01 -> 2026-01-30
  company_status:
company_status
active          69156
liquidation        44
receivership       14


## Normalise types for analysis

Load a typed view (`sme_typed`) with `lloyds_customer` as a real boolean, so filtering and `== True` work. The CSV on disk stays uniform text.

In [13]:
# --- Analysis-ready load: consistent dtypes & safe comparisons -----------------
# The CSV stores everything as text (uniform "True"/"False"/sentinels). This loads
# a typed view where the boolean flags become real nullable booleans, so you can do
# `sme_typed[sme_typed.lloyds_customer]` or `== True`. Non True/False values
# (e.g. 'error') become <NA> and are reported so nothing is hidden.
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme_test.csv"
sme_typed = pd.read_csv(SME_OUT, dtype=str)

bool_map = {"True": True, "False": False}
for col in ("lloyds_customer", "accounts_overdue"):
    if col in sme_typed.columns:
        odd = sme_typed.loc[sme_typed[col].notna() & ~sme_typed[col].isin(bool_map), col]
        if len(odd):
            print(f"Non True/False {col} values -> <NA>:")
            print(odd.value_counts().to_string(), "\n")
        sme_typed[col] = sme_typed[col].map(bool_map).astype("boolean")
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

for col in ("recent_psc_kind", "recent_charge_status"):
    if col in sme_typed.columns:
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

# `sme_typed` is analysis-ready (e.g. sme_typed[sme_typed.lloyds_customer]).
# The CSV on disk stays canonical text.
sme_typed.head()


lloyds_customer:
lloyds_customer
False    67048
True      2166 

Non True/False accounts_overdue values -> <NA>:
accounts_overdue
unknown    1 

accounts_overdue:
accounts_overdue
False    68184
True      1029
<NA>         1 

recent_psc_kind:
recent_psc_kind
individual-person-with-significant-control          57663
no_psc                                               7129
corporate-entity-person-with-significant-control     4338
legal-person-person-with-significant-control           84 

recent_charge_status:
recent_charge_status
no_charges         57478
outstanding         9636
fully-satisfied     2095
part-satisfied         5 



,com_num,name,sic_code,account_type,accounts_overdue,date_of_creation,company_status,last_accounts_made_up_to,next_accounts_due_on,com_type,town,county,post_code,lloyds_customer,recent_psc_kind,recent_charge_status,recent_charge_created_on,n_charges
0,15073164,NFLECTION ADVISORY LIMITED,70229,total-exemption-full,False,2023-08-15,active,2025-03-31,2026-12-31,NaN,NaN,NaN,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0
1,13522064,NFOGENIE LTD,58290,micro-entity,False,2021-07-21,active,2025-07-31,2027-04-30,NaN,NaN,NaN,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0
2,11006939,NNOV8 LIMITED,62090,micro-entity,False,2017-10-11,active,2025-03-31,2026-12-29,NaN,NaN,NaN,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0
3,SC606050,NSPIRED INVESTMENTS LTD,68209,total-exemption-full,False,2018-08-22,active,2025-08-31,2027-05-31,NaN,NaN,NaN,NaN,False,individual-person-with-significant-control,outstanding,2021-09-01,5
4,11303802,"""1ST RATE"" PSYCHOLOGY SERVICES LTD",85600,micro-entity,False,2018-04-11,active,2025-04-29,2027-01-29,NaN,NaN,NaN,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0


## Repair com_num leading zeros

Realign `com_num` in `com_names_sme.csv` to the canonical values in `company_raw/` (fixes leading zeros dropped by Excel, e.g. `5914136` → `05914136`). Idempotent — run it any time, especially after opening/saving the CSV in a spreadsheet.

In [14]:
# --- Repair com_num: realign leading zeros to the raw originals ------------------
# UK company numbers keep leading zeros (e.g. 05914136). Opening the CSV in Excel
# can silently drop them (05914136 -> 5914136, or float-ify to 5914136.0). This
# remaps com_num back to the canonical values in company_raw/. Idempotent:
# reports 0 changed when the file is already aligned.
RAW_DIR = Path("company_data") / "company_raw"
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme_test.csv"


def _norm(x):
    """Collapse a com_num to a match key: drop a stray '.0', then leading zeros."""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x.lstrip("0")


# canonical map: normalized form -> original com_num (from the raw files)
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")
canon = {_norm(c): c for c in raw["com_num"]}

sme = pd.read_csv(SME_OUT, dtype=str)
before = sme["com_num"].copy()
sme["com_num"] = sme["com_num"].map(lambda x: canon.get(_norm(x), str(x).strip()))

changed = int((before != sme["com_num"]).sum())
unmatched = sme.loc[~sme["com_num"].isin(set(raw["com_num"])), "com_num"]
sme.to_csv(SME_OUT, index=False)
print(f"Aligned com_num -> {SME_OUT}")
print(f"  rows changed: {changed}")
print(f"  not matching any raw com_num: {len(unmatched)}")
if len(unmatched):
    print("  examples:", unmatched.head(5).tolist())


Aligned com_num -> company_data/company_sme_with_acctype/com_names_sme_test.csv
  rows changed: 0
  not matching any raw com_num: 0
